# Lab 05-03 — Decomposition: split multi-hop questions into single-hop chunks

**Track 05 · Query transformation** — a multi-hop question needs facts from 2+ chunks ("Who wrote the song, and when did that writer die?"). Plain top-k embeds the WHOLE question and returns the single most-similar chunk — the other fact's chunk is out of reach, so the candidate set is incomplete before any LLM reads it.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, and faiss directly — no repo component library. Every block of the pipeline is built right here:

```
multi-hop question ──► inline decompose prompt + ChatGroq ──► 2-4 sub-questions
        │                                                        │
        ▼                                                        ▼
   FAISS top-3 (raw question)          each sub-question gets its own FAISS top-3
        │                                                        │
        └──────────────► merged, deduplicated, cut to top-6 ◄────┘
```

That is exactly how the shared components in `src/` work underneath: `src/retrieval/decompose.py` is this same prompt + merge loop, and `src/llms/groq.py` is a thin wrapper around the same `ChatGroq`.

Data: HotpotQA (`hotpot_dev_distractor_v1.json`) — a multi-hop benchmark where each question ships its own 10-paragraph "context" plus gold `supporting_facts` (which paragraphs the answer needs). This lab indexes each question's 10 paragraphs into its own small FAISS store, then compares:

* **PLAIN top-3** — the raw question through the same inner retriever;
* **DECOMPOSED** — the inline decompose retriever: original question AND every sub-question, merged, deduplicated, cut to `DECOMPOSE_TOP_K = 6`.

The three questions are pre-selected: plain top-3 MISSES at least one gold supporting paragraph for every one of them (verified against the data), and decomposition recovers the full supporting set — that is the whole point of the technique, measured on gold labels instead of vibes. The Groq LLM is only the decomposer (one call per question); embeddings stay local BGE.


## Setup

One prerequisite must hold before this notebook will run:

- **hotpotqa on disk** — `Data/corpus/hotpotqa/hotpot_dev_distractor_v1.json` (the dev distractor split), already fetched by the repo's manifest-verified fetchers.
- **`GROQ_API_KEY` in the repo-root `.env`** — the Groq LLM is the *decomposer* (it never embeds); embeddings stay local BGE.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `langchain-groq`, `sentence-transformers`, and `faiss-cpu`. The bootstrap cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu langchain-groq python-dotenv


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

# LangChain + sentence-transformers + faiss — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_groq import ChatGroq  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)

load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `HOTPOTQA_PATH` points at the dev-distractor split already on disk; `QUESTION_IDS` lists three pre-selected multi-hop questions — for each one, plain top-3 misses a gold supporting paragraph while decomposition recovers it (verified against the data + the Groq decomposer before shipping); `TOP_K = 3` is the plain retrieval depth (deliberately small: the whole question can only fit one fact), `DECOMPOSE_TOP_K = 6` the merged depth after original + sub-question retrievals; `LLM_MODEL` names the Groq decomposer (never the embedder); `BGE_MODEL_NAME` pins the local embedder; `N_PARAGRAPHS = 10` is how many context paragraphs hotpotqa provides per question.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
HOTPOTQA_PATH = Path("Data/corpus/hotpotqa/hotpot_dev_distractor_v1.json")
# Three multi-hop questions, pre-selected from hotpotqa: for each one, plain
# top-3 misses a gold supporting paragraph while decomposition recovers it
# (verified against the data + the Groq decomposer before shipping).
QUESTION_IDS = [
    "5a722b8655429971e9dc9329",  # "Who was the writer of These Boots… and who died in 2007?"
    "5a8a3e745542996c9b8d5e70",  # "What is the name for the adventure in Tunnels and Trolls…?"
    "5adf37a95542995ec70e8f97",  # "The 2011–12 VCU Rams… represented VCU which was founded in what year?"
]
TOP_K = 3  # plain retrieval depth (deliberately small: the question can only fit one fact)
DECOMPOSE_TOP_K = 6  # merged depth after original + sub-question retrievals
LLM_MODEL = "llama-3.3-70b-versatile"  # Groq is the *decomposer* LLM, never the embedder
# (Gemini alternative: LLM_MODEL = "gemini-2.5-flash" — needs GOOGLE_API_KEY in .env)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
N_PARAGRAPHS = 10  # hotpotqa provides 10 context paragraphs per question


## 2. Load — hotpotqa questions with their self-contained 10-paragraph contexts

`load_hotpotqa` returns `{qid: question_dict}` for the requested rows — each dict keeps `question`, `answer`, `supporting_facts` (list of `[title, sentence_index]` pairs) and `context` (10 `[title, [sentences…]]` paragraphs). `distinct_supporting_titles` deduplicates the gold `supporting_facts` titles in first-seen order — those titles are the ground truth the lab measures both retrievers against. `preview` flattens a paragraph onto one line for printing.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — hotpotqa questions with their self-contained 10-paragraph contexts
# --------------------------------------------------------------------------
def load_hotpotqa(path: Path, ids: list[str]) -> dict[str, dict]:
    """Return {qid: question_dict} for the requested hotpotqa rows.

    Each dict keeps ``question``, ``answer``, ``supporting_facts`` (list of
    [title, sentence_index] pairs) and ``context`` (10 [title, [sentences…]]
    paragraphs). The gold ``supporting_facts`` titles are the ground truth
    the lab measures both retrievers against.
    """
    by_id: dict[str, dict] = {}
    with open(path) as f:
        for q in json.load(f):  # hotpotqa is one JSON array, not JSON-lines
            if q["_id"] in ids:
                by_id[q["_id"]] = q
    return by_id


def distinct_supporting_titles(q: dict) -> list[str]:
    """Gold supporting paragraph titles, deduplicated, in first-seen order."""
    return list(dict.fromkeys(s[0] for s in q["supporting_facts"]))


def preview(text: str, limit: int = 62) -> str:
    """Flatten a paragraph for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3. Experiment — per question: plain top-3 vs decomposed retrieval

The whole pipeline is built inline. The embedder is `HuggingFaceEmbeddings` with the local BGE model (`normalize_embeddings=True`, which BGE requires for cosine). Each question gets its own small store over its 10 paragraphs — chunks built from `" ".join(sentences)` with the paragraph title as metadata, vectors precomputed once and handed to FAISS through a tiny passthrough so embed and index stay separately timed, exactly like the lab. The inner retriever is a plain `similarity_search_by_vector` at `TOP_K = 3` (the inline shape of `src/retrieval/similarity.py`), and the decomposition block is the same `DECOMPOSE_PROMPT` + `ChatGroq` the shared `DecomposeRetriever` wraps — including its safety net: lines that merely restate the original question are dropped, and an empty/failed LLM output falls back to retrieving with the original question alone.

The LLM section reports **run/skip** explicitly: with `GROQ_API_KEY` in `.env` it runs (`ChatGroq`, one call per question); without the key it prints SKIP and decomposition degenerates to plain retrieval — the same contract the lab's `GroqLLM` follows when the key is missing.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — per question: plain top-3 vs decomposed retrieval
# --------------------------------------------------------------------------
DECOMPOSE_PROMPT = """You are a question decomposer for a RAG system.

Given a multi-hop user question, split it into 2-4 INDEPENDENT sub-questions
whose answers together answer the original question.

Example:
  Question: "Who wrote the book Waiting, and where were they born?"
  Sub-questions:
  Who wrote the book Waiting?
  Where was that author born?

Rules:
- Each sub-question must be answerable on its own by one retrieved chunk.
- Do not include the original question.
- Output only the sub-questions, one per line, nothing else.

Question: {question}
Sub-questions:"""


class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order)."""

    def __init__(self, texts: list[str], embeddings: list[list[float]]):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        raise NotImplementedError("precomputed embeddings cannot embed queries")


class _ChatGroqLLM:
    """Inline stand-in for src/llms/groq.GroqLLM: ChatGroq + invoke(str) -> str."""

    def __init__(self, model: str, temperature: float = 0.0):
        self.model = model
        self.temperature = temperature
        self._llm = ChatGroq(model=model, temperature=temperature)

    def invoke(self, prompt: str) -> str:
        return self._llm.invoke(prompt).content


class _SimilarityRetriever:
    """Inline stand-in for src/retrieval/similarity.SimilarityRetriever."""

    def __init__(self, store, embedder, top_k: int = TOP_K):
        self.store = store
        self.embedder = embedder
        self.top_k = top_k

    def retrieve(self, question: str) -> list[Document]:
        """Embed the question and return the top-k most similar documents."""
        query_embedding = self.embedder.embed_query(question)
        return self.store.similarity_search_by_vector(query_embedding, k=self.top_k)


class _DecomposeRetriever:
    """Inline stand-in for src/retrieval/decompose.DecomposeRetriever."""

    def __init__(self, decomposer_llm, retriever, top_k: int = DECOMPOSE_TOP_K):
        self.decomposer_llm = decomposer_llm
        self.retriever = retriever
        self.top_k = top_k

    def _decompose(self, question: str) -> list[str]:
        """Split the question into independent sub-questions via the LLM.

        Falls back to ``[question]`` when nothing usable comes back —
        retrieving with the original question alone is still a valid retrieval.
        """
        try:
            out = self.decomposer_llm.invoke(
                DECOMPOSE_PROMPT.format(question=question)
            )
        except Exception:
            return [question]
        sub = [
            line.strip()
            for line in (out or "").splitlines()
            if line.strip()
        ]
        # Drop lines that just repeat the original question (noise from
        # LLMs that ignore the "do not include the original" rule).
        normalized = question.strip().lower()
        sub = [s for s in sub if s.lower() != normalized]
        return sub if sub else [question]

    def retrieve(self, question: str) -> list[Document]:
        """Retrieve with the original + sub-questions, dedupe, return top-k."""
        queries = [question] + self._decompose(question)

        merged: list[Document] = []
        seen: set[str] = set()
        for sub in queries:
            for doc in self.retriever.retrieve(sub):
                key = doc.page_content
                if key and key not in seen:
                    seen.add(key)
                    merged.append(doc)
        return merged[: self.top_k]


def run_experiment() -> dict:
    questions = load_hotpotqa(HOTPOTQA_PATH, QUESTION_IDS)
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )

    if os.getenv("GROQ_API_KEY"):
        decomposer_llm = _ChatGroqLLM(model=LLM_MODEL)
        llm_status = f"run (ChatGroq {LLM_MODEL})"
    else:
        decomposer_llm = None
        llm_status = ("skip (no GROQ_API_KEY in the repo-root .env — "
                      "decomposition degenerates to plain retrieval)")
        print("LLM section: SKIP —", llm_status)

    results = []
    for qid in QUESTION_IDS:
        q = questions[qid]
        distinct = distinct_supporting_titles(q)

        # Each question gets its own small store over its 10 paragraphs.
        chunks = [
            Document(page_content=" ".join(sents), metadata={"title": title})
            for title, sents in q["context"]
        ]
        t0 = time.perf_counter()
        vecs = embedder.embed_documents([c.page_content for c in chunks])
        store = FAISS.from_documents(chunks, embedding=_PrecomputedEmbeddings(
            [c.page_content for c in chunks], vecs
        ))
        index_s = time.perf_counter() - t0

        inner = _SimilarityRetriever(store, embedder, top_k=TOP_K)
        plain_docs = inner.retrieve(q["question"])

        decomposed = _DecomposeRetriever(decomposer_llm, inner, top_k=DECOMPOSE_TOP_K)
        t0 = time.perf_counter()
        sub_questions = decomposed._decompose(q["question"])
        decompose_s = time.perf_counter() - t0
        decomposed_docs = decomposed.retrieve(q["question"])

        results.append(
            {
                "qid": qid,
                "question": q["question"],
                "answer": q["answer"],
                "supporting": distinct,
                "plain_titles": [d.metadata["title"] for d in plain_docs],
                "sub_questions": sub_questions,
                "decomposed_titles": [d.metadata["title"] for d in decomposed_docs],
                "decomposed_first": preview(decomposed_docs[0].page_content),
                "index_s": index_s,
                "decompose_s": decompose_s,
                "n_paragraphs": len(chunks),
            }
        )

    return {"questions": questions, "results": results, "llm_status": llm_status}


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from four angles: the three questions with their gold answers and supporting paragraphs, plus the LLM section's run/skip status; the plain top-3 titles with `[SUPPORTING]` marks and the missing gold paragraphs; the decomposed path — sub-questions with their generation time, the merged titles with the same marks, and whether ALL gold paragraphs were recovered; then a takeaway on why decomposition fixes retrieval for multi-hop questions: one retrieval pass per fact (plus the original), merged, with the gold `supporting_facts` labels showing the win.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 03 — Decomposition: split multi-hop questions into single-hop chunks")
    print(f"{BGE_MODEL_NAME} (local) | HotpotQA | {LLM_MODEL} decomposer")
    print("=" * 66)
    print(f"    LLM section: {exp['llm_status']}")

    print(f"\n[1] Questions ({len(exp['results'])} from hotpotqa, each with its own")
    print("    10-paragraph context + gold supporting facts):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"][:8]}] "{r["question"]}"')
        print(f"      answer: {r['answer']!r}")
        print(f"      gold supporting paragraphs: {r['supporting']}")

    print(f"\n[2] Plain top-{TOP_K} (raw question, one retrieval pass):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"][:8]}] "{r["question"]}"')
        for t in r["plain_titles"]:
            mark = " [SUPPORTING]" if t in r["supporting"] else ""
            print(f"      - {t}{mark}")
        missing = [t for t in r["supporting"] if t not in r["plain_titles"]]
        print(f"      MISSING gold paragraphs: {missing if missing else 'none'}")

    print(f"\n[3] Decomposed (original + sub-questions, merged to top-{DECOMPOSE_TOP_K}):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"][:8]}] "{r["question"]}"')
        print(f"      sub-questions ({r['decompose_s']:.1f}s):")
        for s in r["sub_questions"]:
            print(f"        - {s}")
        for t in r["decomposed_titles"]:
            mark = " [SUPPORTING]" if t in r["supporting"] else ""
            print(f"      - {t}{mark}")
        recovered = all(t in r["decomposed_titles"] for t in r["supporting"])
        print(f"      all gold paragraphs recovered: {recovered}")

    print("\n[4] Takeaway")
    print("    Decomposition fixes retrieval for multi-hop questions: one")
    print("    retrieval pass per fact (plus the original), merged. The gold")
    print("    supporting_facts labels show the win — every question here is")
    print("    one plain top-3 retrieval could NOT answer, and decomposition")
    print("    brings both paragraphs into the candidate set before the LLM")
    print("    ever reads them.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: all 3 questions loaded with their full 10-paragraph contexts, each with >= 2 distinct gold supporting paragraphs; the problem — plain top-3 must MISS at least one gold paragraph (deterministic pure retrieval, which is what makes the multi-hop question worth decomposing); the fix — decomposition generates >= 1 sub-question, none repeating the original verbatim, the merged retrieval recovers EVERY gold paragraph, and the merged titles are deduplicated. The recovery check is the teaching gate: it only passes when decomposition actually widened the candidate set to the facts plain retrieval could not reach. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Structural: all requested questions loaded, with their full contexts.
    checks.append(("all 3 hotpotqa questions loaded",
                   len(exp["results"]) == len(QUESTION_IDS)))
    checks.append(("every question indexes its full 10-paragraph context",
                   all(r["n_paragraphs"] == N_PARAGRAPHS for r in exp["results"])))
    checks.append(("every question has >= 2 distinct gold supporting paragraphs",
                   all(len(r["supporting"]) >= 2 for r in exp["results"])))

    # The problem: plain top-3 must MISS at least one gold paragraph — this
    # is what makes the multi-hop question worth decomposing (deterministic:
    # pure retrieval, no LLM).
    for r in exp["results"]:
        tag = f"Q{r['qid'][:8]}"
        missing = [t for t in r["supporting"] if t not in r["plain_titles"]]
        checks.append((f"{tag} plain top-{TOP_K} misses >= 1 gold paragraph",
                       len(missing) >= 1))

    # The fix: decomposition must generate sub-questions…
    for r in exp["results"]:
        tag = f"Q{r['qid'][:8]}"
        checks.append((f"{tag} generated >= 1 sub-question",
                       len(r["sub_questions"]) >= 1))
        checks.append((f"{tag} no sub-question repeats the original verbatim",
                       all(s.strip().lower() != r["question"].strip().lower()
                           for s in r["sub_questions"])))

    # …and the merged retrieval must recover EVERY gold paragraph. This is
    # the teaching gate: it only passes when decomposition actually widened
    # the candidate set to the facts plain retrieval could not reach.
    for r in exp["results"]:
        tag = f"Q{r['qid'][:8]}"
        recovered = all(t in r["decomposed_titles"] for t in r["supporting"])
        checks.append((f"{tag} decomposition recovers ALL gold paragraphs",
                       recovered))
        checks.append((f"{tag} decomposed titles are deduplicated",
                       len(r["decomposed_titles"]) == len(set(r["decomposed_titles"]))))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Embedding 3 × 10 paragraphs + 3 Groq decomposition calls — a minute or two, no downloads. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The three multi-hop questions: what plain top-3 brought back (and which gold paragraphs it MISSED), then what the decomposed retrieval recovered — sub-questions, merged titles, and the all-recovered verdict per question.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the hotpotqa file is intact and the LLM section ran (see the status line in the demo).


In [ ]:
verify_gate(exp)
